In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
from mtrain.utils import *
from mtrain.neg_mask.model.datasets.blur_pad_dl import BlurPadDataset

In [ ]:
FOVEATED_DS_PATH = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/training/splits_dataset"
)

SEG_DS_PATH = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/training/seg"
)

In [ ]:
! ls {FOVEATED_DS_PATH}

In [ ]:
import shutil
from tqdm import tqdm

def make_seg_ds_image(imgp: Path):
    maskp = imgp.parent.parent / "masks" / f"{imgp.stem}.png"
    label = BlurPadDataset.label_func(imgp)
    label2cons = {
        "trash": 2,
        "other": 1,
    }
    new_mask = DiskBooleanMask.load(maskp) * label2cons[label]
    return new_mask


def _single_seg_ds(src_root, dest_root):
    imgps = globL(src_root / "train", "*.jpg")
    dest_imgs = mkdir(dest_root / "train")
    dest_masks = mkdir(dest_root / "masks")
    for imgp in tqdm(imgps):
        mask = make_seg_ds_image(imgp)
        shutil.copy(imgp, dest_imgs / imgp.name)
        DiskBooleanMask.save(mask, dest_masks / f"{imgp.stem}.png")

def make_seg_dataset(src_root, dest_root):
    _single_seg_ds(src_root / "train_ds", dest_root / "train_ds")
    _single_seg_ds(src_root / "valid_ds", dest_root / "valid_ds")

In [ ]:
make_seg_dataset(FOVEATED_DS_PATH, SEG_DS_PATH)

In [ ]:
from mtrain.example_dir.learners import step_downer
from mtrain.neg_mask.crops import get_largest_bbox
def get_maskp(imgp):
    return imgp.parent.parent / "masks" / f"{imgp.stem}.png"

def tfm_image(root_ds, dest_ds_name, tfm):
    # pass path to train_ds or valid_ds
    imgps = globL(root_ds / "train", "*.jpg")
    # dest_image_dir = mkdir(root_ds / "step_down_05")
    dest_image_dir = mkdir(root_ds / dest_ds_name)
    for imgp in tqdm(imgps):
        maskp = get_maskp(imgp)
        mask = DiskBooleanMask.load(maskp)
        try:
            bbox = get_largest_bbox(mask)
        except:
            print(f"WARN: error in finding bounding box for {imgp.name}")
            continue
        if bbox is None:
            print(f"WARN: empty mask for {imgp.name}")
            continue
        # img, _ = step_downer(DiskImage.load(imgp), mask, bbox)
        img, _ = tfm(DiskImage.load(imgp), mask, bbox)
        DiskImage.save(img, dest_image_dir / imgp.name)

def add_step_downed_version(ds_path):
    tfm_image(ds_path, "step_down_05", step_downer)

In [ ]:
from mtrain.neg_mask.model.datasets.blur_pad_dl import CropTfmsOutsideBbox

def gaussian_step_down(cropped_image, mask, inner_bbox):
    tfm = CropTfmsOutsideBbox(cropped_image, inner_bbox)
    tfm = tfm.step_down_gaussian(0.5)
    return tfm.crop, mask

def add_gaussian_step_down_version(ds_path):
    tfm_image(ds_path, "gaussian_05", gaussian_step_down)

In [ ]:
add_gaussian_step_down_version(SEG_DS_PATH / "train_ds")
add_gaussian_step_down_version(SEG_DS_PATH / "valid_ds")

In [ ]:
add_step_downed_version(SEG_DS_PATH / "train_ds")
add_step_downed_version(SEG_DS_PATH / "valid_ds")

In [ ]:
imgps = globL(SEG_DS_PATH / "valid_ds" / "gaussian_05", "*.jpg")
len(imgps), imgps[0]

In [ ]:

imgp = imgps[5]
image = DiskImage.load(imgp)
mask = DiskBooleanMask.load(get_maskp(imgp))
print(imgp.name)
print(np.unique(mask))
show([image, mask])